# 01. Build variant-level feature matrix

This notebook starts from the already prepared `master_snv_table_v1.csv` and creates several variant-level matrices for clustering.

Main idea:

- one row = one mtDNA variant
- `variant_id` = variant identifier
- modeling columns = numeric features only
- labels such as `validation_label`, `is_pathogenic_dataset9`, `is_neutral_dataset8`, and `is_artifact_prone_site` are not used in the primary unsupervised matrix

## 1. Imports and paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

INPUT_PATH = Path("../data/processed/master_snv_table_v1.tsv")
OUTPUT_DIR = Path("../data/processed/neutral_scoring_prepared")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_TABLE = OUTPUT_DIR / "neutral_scoring_input_v1.tsv"
FEATURE_SETS_REPORT = OUTPUT_DIR / "neutral_scoring_feature_sets_v1.txt"
LABEL_AUDIT = OUTPUT_DIR / "neutral_scoring_label_audit_v1.tsv"

## 2. Load master file

In [11]:
master_table = pd.read_csv(INPUT_PATH, sep='\t', low_memory=False)
master_table.head()

,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score,gnomad_observed,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_combined_af_simple,gnomad_max_heteroplasmy,hap_defining_variant,vep,feature,gene,helix_counts_hom,helix_af_hom,helix_counts_het,helix_af_het,helix_mean_arf,helix_max_arf,helix_haplogroups_hom,helix_haplogroups_het,helix_observed,is_disease_suspected_dataset3,is_neutral_dataset8,is_pathogenic_dataset9,validation_label,gene_constraint_symbol,gene_constraint_start_position,gene_constraint_end_position,gene_constraint_consequence,gene_constraint_observed,gene_constraint_expected,gene_constraint_obs_exp,gene_constraint_lower_ci,gene_constraint_upper_ci,regional_constraint_symbol,regional_constraint_start_position,regional_constraint_end_position,regional_constraint_protein_residue_start,regional_constraint_protein_residue_end,regional_constraint_observed,regional_constraint_expected,regional_constraint_obs_exp,regional_constraint_lower_ci,regional_constraint_upper_ci,in_regional_constraint,noncoding_constraint_locus,noncoding_constraint_description,noncoding_constraint_start_position,noncoding_constraint_end_position,noncoding_constraint_observed,noncoding_constraint_expected,noncoding_constraint_obs_exp,noncoding_constraint_lower_ci,noncoding_constraint_upper_ci,in_noncoding_constraint,is_artifact_prone_site
0,1,G,T,intergenic_variant,0.65755,m.1G>T,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,1,G,A,intergenic_variant,0.65755,m.1G>A,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,1,G,C,intergenic_variant,0.65755,m.1G>C,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,2,A,T,intergenic_variant,0.64832,m.2A>T,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,2,A,C,intergenic_variant,0.64832,m.2A>C,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [ ]:
print("Shape:", master_table.shape)

Shape: (49704, 64)

Columns:


In [ ]:
id_cols = ["variant_id", "position", "reference", "alternate"]

label_leakage_cols = ["validation_label", "is_disease_suspected_dataset3", "is_neutral_dataset8", "is_pathogenic_dataset9"]

required_basic_cols = id_cols + label_leakage_cols

missing_basic_cols = [col for col in required_basic_cols if col not in master_table.columns]

if missing_basic_cols:

    raise ValueError(f"Missing expected basic columns: {missing_basic_cols}")

# ============================================================

# 4. Binary supervised target

# ============================================================

# На первом этапе используем только:

# neutral -> 0

# pathogenic_confirmed -> 1

#

# disease_suspected и unlabeled пока исключаем из supervised training.

print("\nOriginal validation_label distribution:")

print(master_table[target_col].value_counts(dropna=False))

binary_label_map = {

    "neutral": 0,

    "pathogenic_confirmed": 1,

}

df_cls = master_table[master_table[target_col].isin(binary_label_map.keys())].copy()

df_cls["target"] = df_cls[target_col].map(binary_label_map).astype(int)

print("\nBinary classification subset shape:", df_cls.shape)

print("\nBinary target distribution:")

print(df_cls["target"].value_counts())

print("\nBinary target distribution with labels:")

print(df_cls[target_col].value_counts())


Original validation_label distribution:
validation_label
unlabeled               41280
neutral                  8183
disease_suspected         153
pathogenic_confirmed       88
Name: count, dtype: int64

Binary classification subset shape: (8271, 65)

Binary target distribution:
target
0    8183
1      88
Name: count, dtype: int64

Binary target distribution with labels:
validation_label
neutral                 8183
pathogenic_confirmed      88
Name: count, dtype: int64


In [22]:
# ============================================================
# 5. Feature groups
# ============================================================

mlc_cols = [
    "mlc_score",
]

gnomad_cols = [
    "gnomad_observed",
    "AN",
    "gnomad_homoplasmic_ac",
    "gnomad_heteroplasmic_ac",
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "gnomad_max_heteroplasmy",
]

helix_cols = [
    "helix_observed",
    "helix_counts_hom",
    "helix_af_hom",
    "helix_counts_het",
    "helix_af_het",
    "helix_mean_arf",
    "helix_max_arf",
]

basic_annotation_categorical_cols = [
    "consequence",
    "feature",
    "gene",
]

gene_constraint_numeric_cols = [
    "gene_constraint_start_position",
    "gene_constraint_end_position",
    "gene_constraint_observed",
    "gene_constraint_expected",
    "gene_constraint_obs_exp",
    "gene_constraint_lower_ci",
    "gene_constraint_upper_ci",
]

gene_constraint_categorical_cols = [
    "gene_constraint_symbol",
    "gene_constraint_consequence",
]

regional_constraint_numeric_cols = [
    "regional_constraint_start_position",
    "regional_constraint_end_position",
    "regional_constraint_protein_residue_start",
    "regional_constraint_protein_residue_end",
    "regional_constraint_observed",
    "regional_constraint_expected",
    "regional_constraint_obs_exp",
    "regional_constraint_lower_ci",
    "regional_constraint_upper_ci",
]

regional_constraint_categorical_cols = [
    "regional_constraint_symbol",
]

regional_constraint_flag_cols = [
    "in_regional_constraint",
]

noncoding_constraint_numeric_cols = [
    "noncoding_constraint_start_position",
    "noncoding_constraint_end_position",
    "noncoding_constraint_observed",
    "noncoding_constraint_expected",
    "noncoding_constraint_obs_exp",
    "noncoding_constraint_lower_ci",
    "noncoding_constraint_upper_ci",
]

noncoding_constraint_categorical_cols = [
    "noncoding_constraint_locus",
]

noncoding_constraint_flag_cols = [
    "in_noncoding_constraint",
]

artifact_cols = [
    "is_artifact_prone_site",
]

In [23]:
# ============================================================
# 6. Validate selected columns
# ============================================================

all_selected_source_cols = (
    mlc_cols
    + gnomad_cols
    + helix_cols
    + basic_annotation_categorical_cols
    + gene_constraint_numeric_cols
    + gene_constraint_categorical_cols
    + regional_constraint_numeric_cols
    + regional_constraint_categorical_cols
    + regional_constraint_flag_cols
    + noncoding_constraint_numeric_cols
    + noncoding_constraint_categorical_cols
    + noncoding_constraint_flag_cols
    + artifact_cols
)

missing_selected_cols = [
    col for col in all_selected_source_cols
    if col not in df_cls.columns
]

if missing_selected_cols:
    raise ValueError(f"Missing selected feature columns: {missing_selected_cols}")

In [24]:
# ============================================================
# 7. Convert numeric columns
# ============================================================

numeric_source_cols = (
    mlc_cols
    + gnomad_cols
    + helix_cols
    + gene_constraint_numeric_cols
    + regional_constraint_numeric_cols
    + regional_constraint_flag_cols
    + noncoding_constraint_numeric_cols
    + noncoding_constraint_flag_cols
    + artifact_cols
)

for col in numeric_source_cols:
    df_cls[col] = pd.to_numeric(df_cls[col], errors="coerce")

In [25]:
# ============================================================
# 8. Handle gnomAD and Helix population features
# ============================================================

gnomad_zero_if_absent_cols = [
    "AN",
    "gnomad_homoplasmic_ac",
    "gnomad_heteroplasmic_ac",
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "gnomad_max_heteroplasmy",
]

helix_zero_if_absent_cols = [
    "helix_counts_hom",
    "helix_af_hom",
    "helix_counts_het",
    "helix_af_het",
    "helix_mean_arf",
    "helix_max_arf",
]

df_cls["gnomad_observed"] = df_cls["gnomad_observed"].fillna(0).astype(int)
df_cls["helix_observed"] = df_cls["helix_observed"].fillna(0).astype(int)

df_cls[gnomad_zero_if_absent_cols] = (
    df_cls[gnomad_zero_if_absent_cols]
    .fillna(0.0)
)

df_cls[helix_zero_if_absent_cols] = (
    df_cls[helix_zero_if_absent_cols]
    .fillna(0.0)
)

In [26]:
# ============================================================
# 9. Derived population features
# ============================================================

df_cls["observed_in_any_population_db"] = (
    (df_cls["gnomad_observed"] == 1)
    | (df_cls["helix_observed"] == 1)
).astype(int)

df_cls["observed_in_both_population_dbs"] = (
    (df_cls["gnomad_observed"] == 1)
    & (df_cls["helix_observed"] == 1)
).astype(int)

df_cls["pop_hom_ac_sum"] = (
    df_cls["gnomad_homoplasmic_ac"]
    + df_cls["helix_counts_hom"]
)

df_cls["pop_het_ac_sum"] = (
    df_cls["gnomad_heteroplasmic_ac"]
    + df_cls["helix_counts_het"]
)

df_cls["pop_ac_sum"] = (
    df_cls["pop_hom_ac_sum"]
    + df_cls["pop_het_ac_sum"]
)

df_cls["pop_af_hom_max"] = df_cls[
    ["gnomad_homoplasmic_af", "helix_af_hom"]
].max(axis=1)

df_cls["pop_af_het_max"] = df_cls[
    ["gnomad_heteroplasmic_af", "helix_af_het"]
].max(axis=1)

df_cls["pop_af_max"] = df_cls[
    [
        "gnomad_homoplasmic_af",
        "gnomad_heteroplasmic_af",
        "helix_af_hom",
        "helix_af_het",
    ]
].max(axis=1)

df_cls["pop_af_hom_sum"] = (
    df_cls["gnomad_homoplasmic_af"]
    + df_cls["helix_af_hom"]
)

df_cls["pop_af_het_sum"] = (
    df_cls["gnomad_heteroplasmic_af"]
    + df_cls["helix_af_het"]
)

df_cls["pop_af_sum"] = (
    df_cls["pop_af_hom_sum"]
    + df_cls["pop_af_het_sum"]
)

df_cls["pop_max_heteroplasmy_or_arf"] = df_cls[
    [
        "gnomad_max_heteroplasmy",
        "helix_mean_arf",
        "helix_max_arf",
    ]
].max(axis=1)

In [27]:
# ============================================================
# 10. Log-transformed population features
# ============================================================

eps = 1e-12

log_transform_cols = [
    "gnomad_homoplasmic_ac",
    "gnomad_heteroplasmic_ac",
    "helix_counts_hom",
    "helix_counts_het",
    "pop_hom_ac_sum",
    "pop_het_ac_sum",
    "pop_ac_sum",

    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "helix_af_hom",
    "helix_af_het",
    "pop_af_hom_max",
    "pop_af_het_max",
    "pop_af_max",
    "pop_af_hom_sum",
    "pop_af_het_sum",
    "pop_af_sum",
]

for col in log_transform_cols:
    df_cls[f"log10_{col}"] = np.log10(df_cls[col] + eps)

In [28]:
# ============================================================
# 11. Constraint missing flags and imputation
# ============================================================

df_cls["in_gene_constraint"] = df_cls["gene_constraint_symbol"].notna().astype(int)

constraint_numeric_cols = (
    gene_constraint_numeric_cols
    + regional_constraint_numeric_cols
    + noncoding_constraint_numeric_cols
)

for col in constraint_numeric_cols:
    df_cls[f"{col}_missing"] = df_cls[col].isna().astype(int)
    df_cls[col] = df_cls[col].fillna(0.0)

df_cls["in_regional_constraint"] = (
    df_cls["in_regional_constraint"]
    .fillna(0)
    .astype(int)
)

df_cls["in_noncoding_constraint"] = (
    df_cls["in_noncoding_constraint"]
    .fillna(0)
    .astype(int)
)

df_cls["is_artifact_prone_site"] = (
    df_cls["is_artifact_prone_site"]
    .fillna(0)
    .astype(int)
)

/var/folders/zp/hykk29rd2lndslkkjgkzfb640000gn/T/ipykernel_7272/1142616723.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_cls[f"{col}_missing"] = df_cls[col].isna().astype(int)


In [29]:
# ============================================================
# 12. One-hot encode selected categorical features
# ============================================================

categorical_for_ohe = [
    "consequence",
    "feature",
    "gene",
    "gene_constraint_consequence",
]

for col in categorical_for_ohe:
    df_cls[col] = df_cls[col].fillna("missing").astype(str)

ohe_df = pd.get_dummies(
    df_cls[categorical_for_ohe],
    prefix=categorical_for_ohe,
    dummy_na=False,
    dtype=int,
)

print("\nOne-hot encoded shape:", ohe_df.shape)


One-hot encoded shape: (8271, 71)


In [30]:
# ============================================================
# 13. Core feature set
# ============================================================

core_feature_cols = [
    # Main score
    "mlc_score",

    # Observed flags
    "gnomad_observed",
    "helix_observed",
    "observed_in_any_population_db",
    "observed_in_both_population_dbs",

    # Raw population counts
    "AN",
    "gnomad_homoplasmic_ac",
    "gnomad_heteroplasmic_ac",
    "helix_counts_hom",
    "helix_counts_het",

    # Raw population AF
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "helix_af_hom",
    "helix_af_het",

    # Heteroplasmy / ARF
    "gnomad_max_heteroplasmy",
    "helix_mean_arf",
    "helix_max_arf",
    "pop_max_heteroplasmy_or_arf",

    # Aggregated counts
    "pop_hom_ac_sum",
    "pop_het_ac_sum",
    "pop_ac_sum",

    # Aggregated AF
    "pop_af_hom_max",
    "pop_af_het_max",
    "pop_af_max",
    "pop_af_hom_sum",
    "pop_af_het_sum",
    "pop_af_sum",

    # Log counts
    "log10_gnomad_homoplasmic_ac",
    "log10_gnomad_heteroplasmic_ac",
    "log10_helix_counts_hom",
    "log10_helix_counts_het",
    "log10_pop_hom_ac_sum",
    "log10_pop_het_ac_sum",
    "log10_pop_ac_sum",

    # Log AF
    "log10_gnomad_homoplasmic_af",
    "log10_gnomad_heteroplasmic_af",
    "log10_gnomad_combined_af_simple",
    "log10_helix_af_hom",
    "log10_helix_af_het",
    "log10_pop_af_hom_max",
    "log10_pop_af_het_max",
    "log10_pop_af_max",
    "log10_pop_af_hom_sum",
    "log10_pop_af_het_sum",
    "log10_pop_af_sum",
]

In [31]:
# ============================================================
# 14. Extended feature set
# ============================================================

constraint_missing_flag_cols = [
    f"{col}_missing"
    for col in constraint_numeric_cols
]

extended_numeric_feature_cols = (
    core_feature_cols
    + [
        # Constraint flags
        "in_gene_constraint",
        "in_regional_constraint",
        "in_noncoding_constraint",

        # Artifact flag
        "is_artifact_prone_site",
    ]
    + gene_constraint_numeric_cols
    + regional_constraint_numeric_cols
    + noncoding_constraint_numeric_cols
    + constraint_missing_flag_cols
)

# Remove possible duplicates while preserving order
extended_numeric_feature_cols = list(dict.fromkeys(extended_numeric_feature_cols))

ohe_feature_cols = list(ohe_df.columns)

extended_feature_cols = extended_numeric_feature_cols + ohe_feature_cols

In [32]:
# ============================================================
# 15. Assemble and save core table
# ============================================================

core_output_cols = id_cols + ["target", target_col] + core_feature_cols

core_df = df_cls[core_output_cols].copy()

# Final numeric safety check
for col in core_feature_cols:
    core_df[col] = pd.to_numeric(core_df[col], errors="coerce")

core_missing = core_df[core_feature_cols].isna().sum()
core_missing = core_missing[core_missing > 0]

print("\nCore table shape:", core_df.shape)

print("\nMissing values in core features:")
if len(core_missing) == 0:
    print("No missing values.")
else:
    print(core_missing)

core_df.to_csv(CORE_OUTPUT, sep="\t", index=False)

print("\nSaved core table:")
print(CORE_OUTPUT)


Core table shape: (8271, 52)

Missing values in core features:
No missing values.

Saved core table:
../data/classification_prepared/classification_master_binary_core_v2.tsv


In [33]:
# ============================================================
# 16. Assemble and save extended table
# ============================================================

extended_base_cols = id_cols + ["target", target_col]

extended_df = pd.concat(
    [
        df_cls[extended_base_cols + extended_numeric_feature_cols].copy(),
        ohe_df.copy(),
    ],
    axis=1,
)

# Final numeric safety check
for col in extended_feature_cols:
    extended_df[col] = pd.to_numeric(extended_df[col], errors="coerce")

extended_missing = extended_df[extended_feature_cols].isna().sum()
extended_missing = extended_missing[extended_missing > 0]

print("\nExtended table shape:", extended_df.shape)

print("\nMissing values in extended features:")
if len(extended_missing) == 0:
    print("No missing values.")
else:
    print(extended_missing)

extended_df.to_csv(EXTENDED_OUTPUT, sep="\t", index=False)

print("\nSaved extended table:")
print(EXTENDED_OUTPUT)


Extended table shape: (8271, 173)

Missing values in extended features:
No missing values.

Saved extended table:
../data/classification_prepared/classification_master_binary_extended_v2.tsv


In [34]:
# ============================================================
# 17. Feature report helper
# ============================================================

def make_feature_report(data, feature_cols, output_path):
    rows = []

    for col in feature_cols:
        s = data[col]

        rows.append({
            "feature": col,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "missing_fraction": float(s.isna().mean()),
            "n_unique": int(s.nunique(dropna=True)),
            "min": float(s.min(skipna=True)) if s.notna().any() else np.nan,
            "max": float(s.max(skipna=True)) if s.notna().any() else np.nan,
            "mean": float(s.mean(skipna=True)) if s.notna().any() else np.nan,
        })

    report = pd.DataFrame(rows)
    report.to_csv(output_path, sep="\t", index=False)

    return report


core_report = make_feature_report(
    core_df,
    core_feature_cols,
    CORE_FEATURE_REPORT,
)

extended_report = make_feature_report(
    extended_df,
    extended_feature_cols,
    EXTENDED_FEATURE_REPORT,
)

print("\nSaved core feature report:")
print(CORE_FEATURE_REPORT)

print("\nSaved extended feature report:")
print(EXTENDED_FEATURE_REPORT)


Saved core feature report:
../data/classification_prepared/classification_core_features_report_v2.tsv

Saved extended feature report:
../data/classification_prepared/classification_extended_features_report_v2.tsv


In [35]:
# ============================================================
# 18. Save column sets report
# ============================================================

with open(COLUMN_SETS_REPORT, "w") as f:
    f.write("Target\n")
    f.write("======\n")
    f.write(f"{target_col} -> target\n")

    f.write("\nID columns\n")
    f.write("==========\n")
    for col in id_cols:
        f.write(f"{col}\n")

    f.write("\nLeakage columns excluded from features\n")
    f.write("======================================\n")
    for col in label_leakage_cols:
        f.write(f"{col}\n")

    f.write("\nExplicitly excluded score columns\n")
    f.write("=================================\n")
    f.write("mlc_position_score\n")

    f.write("\nCore feature columns\n")
    f.write("====================\n")
    for col in core_feature_cols:
        f.write(f"{col}\n")

    f.write("\nExtended numeric feature columns\n")
    f.write("===============================\n")
    for col in extended_numeric_feature_cols:
        f.write(f"{col}\n")

    f.write("\nOne-hot encoded categorical source columns\n")
    f.write("==========================================\n")
    for col in categorical_for_ohe:
        f.write(f"{col}\n")

    f.write("\nOne-hot encoded feature columns\n")
    f.write("===============================\n")
    for col in ohe_feature_cols:
        f.write(f"{col}\n")

print("\nSaved column sets report:")
print(COLUMN_SETS_REPORT)


Saved column sets report:
../data/classification_prepared/classification_column_sets_v2.txt
